# 🎙️ VietNeu Voice Studio - Google Colab Edition
Công cụ đọc văn bản & clone giọng nói tiếng Việt bằng AI, chạy trên GPU miễn phí của Google Colab.

### ⚡ Hướng dẫn nhanh:
1. Vào **Thời gian chạy (Runtime)** -> **Thay đổi loại thời gian chạy (Change runtime type)** -> Chọn **T4 GPU** -> Bấm **Lưu (Save)**.
2. Bấm nút **Play (Chạy)** ở Cell 1 để cài thư viện, sau đó bấm chạy Cell 2 để khởi động web app.
3. Cuối log sẽ xuất hiện đường link công khai: `https://xxxx.gradio.live` -> Nhấp vào để mở trên điện thoại!

In [ ]:
# 1. Cài đặt các thư viện cần thiết
!pip install -q vieneu soundfile gradio

In [ ]:
# 2. Khởi động Web App VietNeu Voice Studio (Có tìm kiếm giọng Adam, Hải Đăng...)
import os
import time
import numpy as np
import gradio as gr
from vieneu import Vieneu

VOICES_DIR = "/content/voices"
os.makedirs(VOICES_DIR, exist_ok=True)

print("⏳ Đang nạp mô hình VieNeu-TTS lên GPU Colab...")
tts = Vieneu()
SR = getattr(tts, "sample_rate", 48000)
print(f"✅ Mô hình đã sẵn sàng! Tần số mẫu: {SR} Hz")

# Lấy danh sách toàn bộ 25 giọng chuẩn mặc định của VieNeu
def get_all_preset_voices():
    try:
        raw_voices = tts.list_preset_voices()
        all_v = []
        for label, vid in raw_voices:
            clean_label = label.replace("⭐ ", "").replace("⭐", "").strip()
            all_v.append({"display": clean_label, "name": vid})
        return all_v
    except Exception:
        return [
            {"display": "Adam — Nam · Nam · Giọng đọc tự nhiên", "name": "Adam"},
            {"display": "Adam bựa — Nam · Bắc · Phong cách tự nhiên", "name": "Adam bựa"},
            {"display": "Hải Đăng — Nam · Bắc · Phong cách tự nhiên", "name": "Hải Đăng"},
        ]

ALL_PRESET_VOICES = get_all_preset_voices()
VOICE_MAP = {v["display"]: v["name"] for v in ALL_PRESET_VOICES}
ALL_DISPLAYS = [v["display"] for v in ALL_PRESET_VOICES]
DEFAULT_DISPLAY = next((d for d in ALL_DISPLAYS if "Adam —" in d), ALL_DISPLAYS[0] if ALL_DISPLAYS else "")

# Hàm lọc tìm kiếm giọng đọc theo từ khóa và phân loại
def filter_preset_voices(search_text, category):
    q = (search_text or "").strip().lower()
    filtered = []
    for v in ALL_PRESET_VOICES:
        disp = v["display"]
        disp_lower = disp.lower()
        if category == "Giọng Nam" and "— Nam" not in disp and "· Nam" not in disp:
            continue
        elif category == "Giọng Nữ" and "— Nữ" not in disp and "· Nữ" not in disp:
            continue
        elif category == "Miền Bắc" and "Bắc" not in disp:
            continue
        elif category == "Miền Nam" and ("· Nam ·" not in disp and "— Nam · Nam" not in disp and "— Nữ · Nam" not in disp):
            continue
        elif category == "Miền Trung" and "Trung" not in disp:
            continue
        if q and q not in disp_lower:
            continue
        filtered.append(disp)
        
    if not filtered:
        return gr.Dropdown(choices=["(Không tìm thấy giọng phù hợp)"], value="(Không tìm thấy giọng phù hợp)")
    return gr.Dropdown(choices=filtered, value=filtered[0])

def get_custom_voice_files():
    valid_exts = (".wav", ".mp3", ".ogg", ".flac", ".m4a")
    if not os.path.exists(VOICES_DIR):
        return []
    return sorted([f for f in os.listdir(VOICES_DIR) if f.lower().endswith(valid_exts)])

SAMPLE_TEXT = "Xin chào bạn! Tôi là Adam. Đây là công cụ chuyển đổi văn bản thành giọng nói tiếng Việt tự động chạy trên GPU Google Colab."

def process_tts(text, mode, preset_voice_display, custom_voice_file, temperature):
    text = (text or "").strip()
    if not text:
        raise gr.Error("⚠️ Vui lòng nhập nội dung văn bản cần đọc!")

    start_time = time.perf_counter()

    if mode == "preset":
        if not preset_voice_display or "Không tìm thấy" in preset_voice_display:
            raise gr.Error("⚠️ Vui lòng chọn một giọng trong danh sách!")
        actual_voice_name = VOICE_MAP.get(preset_voice_display, preset_voice_display.split(" — ")[0].strip())
        audio = tts.infer(text, voice=actual_voice_name, temperature=float(temperature))
        info_source = f"Giọng chuẩn VieNeu: **{actual_voice_name}**"
    else:
        if not custom_voice_file or "Chưa có file" in custom_voice_file:
            raise gr.Error("⚠️ Thư mục '/content/voices' chưa có file .wav nào! Hãy tải file .wav (3-5s) vào thư mục voices bên cột trái Colab.")
        ref_path = os.path.join(VOICES_DIR, custom_voice_file)
        if not os.path.exists(ref_path):
            raise gr.Error(f"⚠️ Không tìm thấy file: {custom_voice_file}")
        audio = tts.infer(text, ref_audio=ref_path, temperature=float(temperature))
        info_source = f"Clone từ file: **{custom_voice_file}**"

    elapsed = time.perf_counter() - start_time
    audio_data = np.asarray(audio, dtype=np.float32)
    duration = len(audio_data) / SR if SR else 0.0

    stats = (
        f"✅ **Đã tạo giọng nói thành công!**\n\n"
        f"• **Nguồn giọng:** {info_source}\n"
        f"• **Thời gian xử lý:** **{elapsed:.2f}s** (Siêu tốc trên GPU)\n"
        f"• **Độ dài audio:** **{duration:.2f}s**"
    )
    return (SR, audio_data), stats

def refresh_file_list():
    files = get_custom_voice_files()
    if not files:
        return gr.Dropdown(choices=["Chưa có file mẫu trong thư mục voices"], value="Chưa có file mẫu trong thư mục voices")
    return gr.Dropdown(choices=files, value=files[0])

def on_mode_change(mode):
    if mode == "preset":
        return gr.update(visible=True), gr.update(visible=False)
    else:
        return gr.update(visible=False), gr.update(visible=True)

custom_css = """
.container { max-width: 680px !important; margin: 0 auto !important; padding: 10px !important; }
.btn-generate { height: 52px !important; font-size: 1.15rem !important; font-weight: 700 !important; border-radius: 12px !important; margin-top: 10px !important; }
.audio-player { border-radius: 12px !important; }
"""

theme = gr.themes.Soft(primary_hue="indigo", secondary_hue="cyan", neutral_hue="slate")

with gr.Blocks(title="VietNeu Voice Studio") as demo:
    with gr.Column(elem_classes=["container"]):
        gr.Markdown(
            """
            <div style="text-align: center; margin-bottom: 12px;">
                <h1 style="font-size: 1.8rem; margin-bottom: 4px;">🎙️ VietNeu Voice Studio (Colab GPU)</h1>
                <p style="color: #64748b; font-size: 0.95rem;">Đọc văn bản & Clone giọng nói tiếng Việt bằng AI</p>
            </div>
            """
        )

        text_input = gr.Textbox(label="📝 1. Nhập văn bản cần đọc", value=SAMPLE_TEXT, lines=4, max_lines=8)
        mode_radio = gr.Radio(choices=[("Kiểu 1: Giọng có sẵn (VieNeu)", "preset"), ("Kiểu 2: Giọng clone từ file (.wav)", "clone")], value="preset", label="🎯 2. Chọn kiểu giọng nói")

        preset_box = gr.Group(visible=True)
        with preset_box:
            with gr.Row():
                search_box = gr.Textbox(label="🔍 Tìm kiếm giọng đọc", placeholder="Gõ để tìm: Adam, Hải Đăng, Quỳnh Anh, Bắc, Nam...", lines=1, scale=4)
                clear_search_btn = gr.Button("✖️ Xoá tìm", scale=1, variant="secondary")
            category_pills = gr.Radio(choices=["Tất cả", "Giọng Nam", "Giọng Nữ", "Miền Bắc", "Miền Nam", "Miền Trung"], value="Tất cả", label="⚡ Lọc nhanh theo tiêu chí")
            preset_dropdown = gr.Dropdown(label="🎙️ Danh sách giọng đọc (Có ô tìm kiếm bên trong)", choices=ALL_DISPLAYS, value=DEFAULT_DISPLAY, filterable=True, interactive=True)

        initial_files = get_custom_voice_files()
        initial_choices = initial_files if initial_files else ["Chưa có file mẫu trong thư mục voices"]
        
        clone_box = gr.Group(visible=False)
        with clone_box:
            with gr.Row():
                custom_dropdown = gr.Dropdown(label="Chọn file giọng mẫu trong thư mục /content/voices/", choices=initial_choices, value=initial_choices[0], allow_custom_value=True, interactive=True, scale=4)
                refresh_btn = gr.Button("🔄 Quét lại", scale=1, variant="secondary")
            gr.Markdown("💡 *Mẹo: Kéo thả file `.wav` (3–5s) vào thư mục `voices` ở cột quản lý file bên trái Colab, rồi bấm **Quét lại**.*")

        generate_btn = gr.Button("🔊 Bắt đầu tạo giọng nói", variant="primary", elem_classes=["btn-generate"])
        audio_output = gr.Audio(label="🎧 Kết quả âm thanh (Nghe & Tải về)", type="numpy", autoplay=True)
        stats_output = gr.Markdown()

        with gr.Accordion("⚙️ Cài đặt nâng cao (Tuỳ chọn)", open=False):
            temperature_slider = gr.Slider(0.1, 1.5, value=0.8, step=0.05, label="Độ biểu cảm (Temperature) - Mặc định 0.8")

    mode_radio.change(fn=on_mode_change, inputs=[mode_radio], outputs=[preset_box, clone_box])
    search_box.input(fn=filter_preset_voices, inputs=[search_box, category_pills], outputs=[preset_dropdown])
    category_pills.change(fn=filter_preset_voices, inputs=[search_box, category_pills], outputs=[preset_dropdown])
    clear_search_btn.click(fn=lambda: ("", "Tất cả", gr.Dropdown(choices=ALL_DISPLAYS, value=DEFAULT_DISPLAY)), outputs=[search_box, category_pills, preset_dropdown])
    refresh_btn.click(fn=refresh_file_list, outputs=[custom_dropdown])
    generate_btn.click(fn=process_tts, inputs=[text_input, mode_radio, preset_dropdown, custom_dropdown, temperature_slider], outputs=[audio_output, stats_output])

demo.queue().launch(share=True, theme=theme, css=custom_css)
